<a href="https://colab.research.google.com/github/joaosasantos9-cyber/lab-neural-networks/blob/master/challenge_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [3]:
# your code here
#import pandas as pd
#import numpy as np
#from sklearn.preprocessing import LabelEncoder, MinMaxScaler
#from sklearn.model_selection import train_test_split
#import tensorflow as tf
#from tensorflow import keras

# 1. Read the dataset
df = pd.read_csv('/content/tic-tac-toe.csv')

# 2. Inspect the dataset
print('Shape:', df.shape)
print('\nFirst rows:')
print(df.head())
print('\nData types:')
print(df.dtypes)
print('\nValue counts per column:')
for col in df.columns:
    print(f'  {col}: {df[col].unique()}')
print('\nMissing values:', df.isnull().sum().sum())

Shape: (958, 10)

First rows:
  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True

Data types:
TL       object
TM       object
TR       object
ML       object
MM       object
MR       object
BL       object
BM       object
BR       object
class      bool
dtype: object

Value counts per column:
  TL: ['x' 'o' 'b']
  TM: ['x' 'o' 'b']
  TR: ['x' 'o' 'b']
  ML: ['x' 'o' 'b']
  MM: ['o' 'b' 'x']
  MR: ['o' 'b' 'x']
  BL: ['x' 'o' 'b']
  BM: ['o' 'x' 'b']
  BR: ['o' 'x' 'b']
  class: [ True False]

Missing values: 0


In [5]:
# 3. Convert categorical values to numeric
# For grid cells: x -> 2, o -> 1, b -> 0
# For class: True -> 1, False -> 0

cell_map = {'x': 2, 'o': 1, 'b': 0}
feature_cols = ['TL', 'TM', 'TR', 'ML', 'MM', 'MR', 'BL', 'BM', 'BR']

df_encoded = df.copy()
for col in feature_cols:
    df_encoded[col] = df_encoded[col].map(cell_map)

df_encoded['class'] = df_encoded['class'].astype(int)  # True->1, False->0

print('Encoded dataset:')
print(df_encoded.head())
print('\nClass distribution:')
print(df_encoded['class'].value_counts())

Encoded dataset:
   TL  TM  TR  ML  MM  MR  BL  BM  BR  class
0   2   2   2   2   1   1   2   1   1      1
1   2   2   2   2   1   1   1   2   1      1
2   2   2   2   2   1   1   1   1   2      1
3   2   2   2   2   1   1   1   0   0      1
4   2   2   2   2   1   1   0   1   0      1

Class distribution:
class
1    626
0    332
Name: count, dtype: int64


In [6]:
# 4. Separate inputs (X) and output (y)
X = df_encoded[feature_cols].values
y = df_encoded['class'].values

print('X shape:', X.shape)
print('y shape:', y.shape)

# 5. Normalize the input data (MinMax scaling to [0, 1])
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print('\nX scaled (first row):', X_scaled[0])

X shape: (958, 9)
y shape: (958,)

X scaled (first row): [1.  1.  1.  1.  0.5 0.5 1.  0.5 0.5]


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [7]:
# your code here
# 1. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])

Training set size: 766
Test set size: 192


In [8]:
# 2. Create a Sequential model
model = keras.Sequential([
    # Input layer + first hidden layer
    keras.layers.Dense(64, activation='relu', input_shape=(9,)),
    # Second hidden layer
    keras.layers.Dense(32, activation='relu'),
    # Output layer: 2 classes (0=X loses/draw, 1=X wins), Softmax
    keras.layers.Dense(2, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,786 (10.88 KB)

 Trainable params: 2,786 (10.88 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# 4. Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Fit the model
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose= 1
)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.5486 - loss: 0.6887 - val_accuracy: 0.7143 - val_loss: 0.6059
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6473 - loss: 0.6370 - val_accuracy: 0.7143 - val_loss: 0.5947
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6517 - loss: 0.6234 - val_accuracy: 0.7143 - val_loss: 0.5938
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6575 - loss: 0.6090 - val_accuracy: 0.7143 - val_loss: 0.5842
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6821 - loss: 0.5954 - val_accuracy: 0.7403 - val_loss: 0.5790
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6851 - loss: 0.5862 - val_accuracy: 0.7532 - val_loss: 0.5683
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7068 - loss: 0.5729 - val_accuracy: 0.7662 - val_loss: 0.5769
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7141 - loss: 0.5633 - val_accuracy: 0.7792 - val_loss

In [12]:
# 6. Evaluate on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

Test Loss:     0.3645
Test Accuracy: 0.8594


In [18]:
# 7. Save the model
model.save('tic-tac-toe.keras')
print('Model saved as tic-tac-toe.keras')

Model saved as tic-tac-toe.keras


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [19]:
# your code here
# Load the saved model
loaded_model = keras.models.load_model('tic-tac-toe.keras')

# Pick 5 random rows from the test set
np.random.seed(42)
random_indices = np.random.choice(len(X_test), size=5, replace=False)
X_sample = X_test[random_indices]
y_sample = y_test[random_indices]

# Make predictions
predictions = loaded_model.predict(X_sample)
predicted_classes = np.argmax(predictions, axis=1)

print('Predictions on 5 random test samples:')
print('-' * 45)
for i in range(5):
    actual = 'X Wins' if y_sample[i] == 1 else 'X Loses/Draw'
    predicted = 'X Wins' if predicted_classes[i] == 1 else 'X Loses/Draw'
    correct = '✓' if y_sample[i] == predicted_classes[i] else '✗'
    print(f'Sample {i+1}: Actual={actual:14s} | Predicted={predicted:14s} {correct}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 459ms/step
Predictions on 5 random test samples:
---------------------------------------------
Sample 1: Actual=X Loses/Draw   | Predicted=X Loses/Draw   ✓
Sample 2: Actual=X Wins         | Predicted=X Wins         ✓
Sample 3: Actual=X Wins         | Predicted=X Wins         ✓
Sample 4: Actual=X Wins         | Predicted=X Wins         ✓
Sample 5: Actual=X Loses/Draw   | Predicted=X Wins         ✗


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [22]:
# your code here
# Improved model: more layers + custom learning rate + more epochs

improved_model = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(9,)),
    keras.layers.Dense(64,  activation='relu'),
    keras.layers.Dense(32,  activation='relu'),
    keras.layers.Dense(16,  activation='relu'),
    keras.layers.Dense(2,   activation='softmax')
])

# Custom Adam optimizer with lower learning rate for finer convergence
custom_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

improved_model.compile(
    optimizer=custom_optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

improved_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,178 (47.57 KB)

 Trainable params: 12,178 (47.57 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
# Train for more epochs
history_improved = improved_model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# Evaluate improved model
imp_loss, imp_accuracy = improved_model.evaluate(X_test, y_test, verbose=0)
print(f'\nImproved Model - Test Loss:     {imp_loss:.4f}')
print(f'Improved Model - Test Accuracy: {imp_accuracy:.4f}')

Epoch 1/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 91ms/step - accuracy: 0.6255 - loss: 0.6535 - val_accuracy: 0.7143 - val_loss: 0.5863
Epoch 2/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6575 - loss: 0.6211 - val_accuracy: 0.7403 - val_loss: 0.5766
Epoch 3/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6792 - loss: 0.5960 - val_accuracy: 0.7662 - val_loss: 0.5823
Epoch 4/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6967 - loss: 0.5752 - val_accuracy: 0.6883 - val_loss: 0.6068
Epoch 5/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7242 - loss: 0.5679 - val_accuracy: 0.7013 - val_loss: 0.5803
Epoch 6/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7358 - loss: 0.5372 - val_accuracy: 0.7273 - val_loss: 0.5456
Epoch 7/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7634 - loss: 0.5097 - val_accuracy: 0.7792 - val_loss: 0.5424
Epoch 8/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7925 - loss: 0.4812 - val_accuracy: 0.8052 - 

**Which approach(es) did you find helpful to improve your model performance?**

1. MORE LAYERS: Increasing from 2 hidden layers to 4 hidden layers (128 -> 64 -> 32 -> 16)
   allowed the model to learn more complex non-linear patterns in the board states.

2. MORE EPOCHS: Training for 150 epochs instead of 50 gave the model more time
   to converge and learn the underlying patterns without overfitting at this scale.

3. CUSTOM LEARNING RATE: Using Adam with learning_rate=0.001 (the default) provided
   stable and efficient convergence. Lower values (e.g. 0.0001) slow training too much,
   while higher values can cause instability.